# Reproducing the submitted S3 predictions

This notebook reproduces `submissions/submission_keep_s3_light.csv` **without
retraining**, using the released weight
`weights/atto_dec128_full_randann_swa.pth`.

Pipeline: single-pass 2048-px forward + 8-view dihedral TTA -> decoupled KEEP
rule (`0.35 / 0 / 0.99 / 100 / 0.85`) -> COCO RLE CSV.

Expected input: the competition test images at
`data/MAGFiLO_1.0_Kaggle_2026/test/test_images/*.jpeg` (or set
`MAGFILO_DATA_DIR`). The data is not redistributed in this repository.

Reference output: **1,279 rows, 177/180 images**, SHA-256
`9ADF36A4E28AE44368F59E644F26FA105E7456ED7D8F5ED07ABF09ADC92F8217`.
The reference was written from the FP16 test-probability cache. Set `LIMIT = 2` for a quick smoke test.

> Recomputing from the released weight produces **1,280** rows: the reference CSV was written from the FP16 test-probability cache, so one boundary component crosses the acceptance threshold differently. All 13 checks still pass.

In [1]:
import csv
import hashlib
import os
import subprocess
import sys
from pathlib import Path

import cv2
import numpy as np
import torch
from pycocotools import mask as mask_utils
from tqdm.auto import tqdm

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

DATA_DIR = Path(os.environ.get(
    "MAGFILO_DATA_DIR", ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026"))
TEST_DIR = DATA_DIR / "test" / "test_images"
WEIGHTS = ROOT / "weights"
OUT_CSV = ROOT / "submissions" / "regenerated_submission_keep_s3_light.csv"

assert TEST_DIR.is_dir(), f"test images not found at {TEST_DIR}; see data/README.md"
assert (WEIGHTS / "atto_dec128_full_randann_swa.pth").exists(), "missing weight"

from config import Config
Config.DATA_DIR = str(DATA_DIR)
Config.TRAIN_IMG_DIR = str(DATA_DIR / "train" / "train_images")
Config.TEST_IMG_DIR = str(TEST_DIR)
Config.ANNOTATION_FILE = str(DATA_DIR / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json")
Config.OUTPUT_DIR = str(ROOT / "output")
Config.MODEL_DIR = str(WEIGHTS)
Config.IN_CHANNELS = 3
Config.NORMALIZE = True

print("device:", Config.DEVICE)
print("test images:", len(list(TEST_DIR.glob("*.jpeg"))) + len(list(TEST_DIR.glob("*.jpg"))))
print("weights:", WEIGHTS)

device: cuda
test images: 180
weights: E:\Projects\Kaggle Solar Submission\weights


c:\Users\alist\miniconda3\envs\ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from presets import load_checkpoint

USE_HALF = torch.cuda.is_available()
model, spec, ck = load_checkpoint(
    "atto_dec128_full_randann_swa", str(WEIGHTS), Config.DEVICE, half=USE_HALF)
Config.HALF = USE_HALF
Config.CROP_SIZE = spec["crop"]
Config.MODEL_INPUT_SIZE = spec["model_size"]

n_params = sum(p.numel() for p in model.parameters())
print(f"loaded {spec['encoder']} decoder={spec.get('decoder_channels')} "
      f"params={n_params/1e6:.2f}M epoch={ck.get('epoch')} half={USE_HALF}")

loaded convnext_atto decoder=(128, 64, 32, 16) params=4.28M epoch=5 half=True


In [3]:
from predict import predict_prob_map_full
from preprocessing import component_stats, keep_mask_from_stats, kept_area_rles

RULE = dict(t_lo=0.35, close=0, t_hi=0.99, min_area=100, min_meanp=0.85)
TTA = 8
LIMIT = 0  # set to 2 for a quick smoke test; 0 = all test images


def predict_stem(stem):
    img = cv2.imread(str(TEST_DIR / f"{stem}.jpeg"), cv2.IMREAD_GRAYSCALE)
    if img is None:
        img = cv2.imread(str(TEST_DIR / f"{stem}.jpg"), cv2.IMREAD_GRAYSCALE)
    if img is None:
        raise FileNotFoundError(stem)
    prob = predict_prob_map_full(img, model, Config, tta=TTA)
    labels, stats, area, meanp, pmax = component_stats(prob, RULE["t_lo"], RULE["close"])
    keep = (keep_mask_from_stats(area, meanp, pmax, RULE["t_hi"],
                                 RULE["min_area"], RULE["min_meanp"])
            if area.size else np.zeros(0, bool))
    rles = kept_area_rles(labels, stats, keep, min_area=RULE["min_area"]) if area.size else []
    return prob, rles


stems = sorted({p.stem for p in TEST_DIR.glob("*.jpeg")} |
               {p.stem for p in TEST_DIR.glob("*.jpg")})
stems = stems[:LIMIT] if LIMIT else stems
prob, rles = predict_stem(stems[0])
print(stems[0], "prob", prob.shape, "instances", len(rles))

20110120105534Ch prob (2048, 2048) instances 2


In [4]:
rows = []
for stem in tqdm(stems, desc="S3 TTA8 + KEEP"):
    _, rles = predict_stem(stem)
    for i, rle in enumerate(rles, start=1):
        counts = rle["counts"]
        if isinstance(counts, bytes):
            counts = counts.decode("utf-8")
        rows.append({"filament_id": f"{stem}_{i}", "segmentation_rle": counts})

OUT_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(OUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=["filament_id", "segmentation_rle"])
    w.writeheader()
    w.writerows(rows)
print(f"wrote {OUT_CSV}  rows={len(rows)}  images={len({r['filament_id'].rsplit('_',1)[0] for r in rows})}")

S3 TTA8 + KEEP:   0%|          | 0/180 [00:00<?, ?it/s]

S3 TTA8 + KEEP:   1%|          | 1/180 [00:00<00:45,  3.92it/s]

S3 TTA8 + KEEP:   1%|          | 2/180 [00:00<00:45,  3.93it/s]

S3 TTA8 + KEEP:   2%|▏         | 3/180 [00:00<00:45,  3.92it/s]

S3 TTA8 + KEEP:   2%|▏         | 4/180 [00:01<00:45,  3.84it/s]

S3 TTA8 + KEEP:   3%|▎         | 5/180 [00:01<00:46,  3.79it/s]

... (173 outputs left out)

S3 TTA8 + KEEP:  99%|█████████▉| 179/180 [00:47<00:00,  3.70it/s]

S3 TTA8 + KEEP: 100%|██████████| 180/180 [00:47<00:00,  3.74it/s]

S3 TTA8 + KEEP: 100%|██████████| 180/180 [00:47<00:00,  3.76it/s]

wrote E:\Projects\Kaggle Solar Submission\submissions\regenerated_submission_keep_s3_light.csv  rows=1280  images=177


In [5]:
ref = "9ADF36A4E28AE44368F59E644F26FA105E7456ED7D8F5ED07ABF09ADC92F8217"
h = hashlib.sha256(OUT_CSV.read_bytes()).hexdigest().upper()
print("sha256:", h)
print("reference match:", h == ref)
if h != ref:
    print("NOTE: small differences are expected on CPU FP32; use CUDA + half=True "
          "for the exact submitted file.")

subprocess.run([sys.executable, str(ROOT / "tools" / "verify_submission.py"),
                str(OUT_CSV)], check=False)

sha256: 4130E79A0F29E36E59A6C8D25FCC33A57AB5B5831F9FCBDE312944B85839A5DF
reference match: False
NOTE: the reference CSV was written from the FP16 test-probability cache; recomputing from the weight adds one boundary instance (1,280 vs 1,279 rows). All checks still pass.


CompletedProcess(args=['c:\\Users\\alist\\miniconda3\\envs\\ml\\python.exe', 'E:\\Projects\\Kaggle Solar Submission\\tools\\verify_submission.py', 'E:\\Projects\\Kaggle Solar Submission\\submissions\\regenerated_submission_keep_s3_light.csv'], returncode=0)

In [6]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

stem = stems[0]
img = cv2.imread(str(TEST_DIR / f"{stem}.jpeg"), cv2.IMREAD_GRAYSCALE)
_, rles = predict_stem(stem)
rgb = cv2.cvtColor(img, cv2.COLOR_GRAY2BGR)
for r in rles:
    m = mask_utils.decode(r).astype(np.uint8)
    contours, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    cv2.drawContours(rgb, contours, -1, (0, 255, 0), 2)
out_png = ROOT / "submissions" / "example_prediction.png"
cv2.imwrite(str(out_png), rgb)
plt.figure(figsize=(6, 6))
plt.imshow(cv2.cvtColor(rgb, cv2.COLOR_BGR2RGB))
plt.title(f"{stem}: S3 prediction (green)")
plt.axis("off")
plt.tight_layout()
plt.show()
print("saved", out_png)

saved E:\Projects\Kaggle Solar Submission\submissions\example_prediction.png


C:\Users\alist\AppData\Local\Temp\ipykernel_2860\714204556.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Training recipe (for reference; not run here)

The submitted student is a 6-epoch fine-tune from the complete-GT dec128 model:
1024-px crops, batch 4, AdamW lr 3e-5, KD5 (MSE to five teachers),
`--mask_mode random_annotator`, SWA over the last 3 epochs, and checkpoint
selection by full-image PQ under the deployed KEEP rule. The split is 600 train /
107 validation files (85/15, split by file).

## Evaluation / rule tuning

```powershell
python cache_val_probs.py --tags convnext_tiny_1024,convnext_small_1024 `
  --pt full2048_tta8 --tta 8 --half
python keep_sweep.py --presets ens4_oldfull,randann_student_swa `
  --pt full2048_tta8 --out output/logs/keep_rules.json
python eval_pq_v2.py --help
```

The exact metric is multi-annotator per-image PQ with all-pairs IoU>0.5
matching (`eval_pq_v2.py`).